# Notebook 2: QLoRA Fine-tuning of TinyLlama-1.1B on Code

**Goal:** Fine-tune TinyLlama-1.1B with QLoRA on CodeSearchNet (Python) so its token distribution aligns with CodeLlama-7B's code domain. This produces the **domain-tuned draft model (Condition 2)**.

**Why this matters:** The acceptance rate in speculative decoding is directly tied to how similar the draft and target distributions are. A generic TinyLlama outputs `the`, `is`, `def` etc. but may not predict code-specific patterns (indentation, brackets, library names) as well as a code-tuned version.

---
**Hardware:** A100 recommended (Colab PAYG). T4 works but is slow (~6hr for 10% corpus).

**Runtime estimates on A100:**
- 10% corpus, 3 epochs → ~45 min
- 50% corpus, 3 epochs → ~3.5 hours
- 100% corpus, 3 epochs → ~7 hours

## 0. Setup

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets wandb

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > A100 GPU")

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Verify Drive is accessible before proceeding
assert os.path.exists('/content/drive/MyDrive'), "Drive mount failed — do not proceed with training!"
print("Google Drive mounted successfully at /content/drive/MyDrive")

In [ ]:
from huggingface_hub import login
import wandb

login()             # HuggingFace token
wandb.login()       # Weights & Biases token (for training curves)

## 1. Configuration
Change `SPLIT_FRACTION` for the dataset-size ablation study:
- `0.1` → 10% corpus (~40K samples, ~20 min)
- `0.5` → 50% corpus (~200K samples, ~1.5 hr)
- `1.0` → 100% corpus (~400K samples, ~3–4 hr)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ▶▶ CHANGE THIS BEFORE EACH RUN ◀◀
SPLIT_FRACTION  = 1.0   # 0.1 = 10% (done ✓) | 0.5 = 50% (done ✓) | 1.0 = 100%
# ══════════════════════════════════════════════════════════════════════

BASE_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MAX_SEQ_LENGTH  = 512
OUTPUT_DIR      = f"/content/drive/MyDrive/tinyllama-qlora-code-{int(SPLIT_FRACTION*100)}pct"
HF_REPO_ID      = f"nishant-k/tinyllama-code-specdraft-{int(SPLIT_FRACTION*100)}pct"

# QLoRA
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training — A100 optimised (batch=16, ~4-5hrs for 50% corpus)
EPOCHS          = 3
BATCH_SIZE      = 16
GRAD_ACCUM      = 2          # effective batch = 32
LR              = 2e-4
WARMUP_RATIO    = 0.05

WANDB_RUN_NAME  = f"tinyllama-qlora-code-{int(SPLIT_FRACTION*100)}pct"

assert SPLIT_FRACTION in [0.1, 0.5, 1.0], "SPLIT_FRACTION must be 0.1, 0.5, or 1.0"
print(f"Config set.")
print(f"  SPLIT_FRACTION : {SPLIT_FRACTION} ({int(SPLIT_FRACTION*100)}% corpus)")
print(f"  Output dir     : {OUTPUT_DIR}")
print(f"  HF repo        : {HF_REPO_ID}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

## 2. Load and Prepare Dataset

In [ ]:
from datasets import load_dataset

print("Loading CodeSearchNet (Python)...")
raw = load_dataset("code_search_net", "python")
train_data = raw["train"]

if SPLIT_FRACTION < 1.0:
    n = int(len(train_data) * SPLIT_FRACTION)
    train_data = train_data.shuffle(seed=42).select(range(n))

print(f"Training samples: {len(train_data):,}")
print(f"\nSample entry:")
print(train_data[0]["whole_func_string"][:300])

In [ ]:
# Format: plain next-token prediction on raw code (no instruction template)
# Goal is to learn code token distribution, not instruction-following
def format_sample(example):
    return {"text": example["whole_func_string"]}

dataset = train_data.map(format_sample, remove_columns=train_data.column_names)
print(f"Dataset ready. Columns: {dataset.column_names}")
print(f"Sample text: {dataset[0]['text'][:200]}")

## 3. Load Model with QLoRA Config

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantized base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Attach LoRA adapters
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5-1% of params are trainable — normal for LoRA r=16

## 4. Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

# Tokenize dataset
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"])
print(f"Tokenized dataset: {len(tokenized_dataset):,} samples")

warmup_steps = int(WARMUP_RATIO * (len(tokenized_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS)

import torch
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"Using {'bf16' if use_bf16 else 'fp16'} training")

# Auto-detect latest checkpoint in Drive for resuming after failure
resume_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = sorted([
        os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ], key=lambda x: int(x.split("-")[-1]))
    if checkpoints:
        resume_checkpoint = checkpoints[-1]
        print(f"Resuming from checkpoint: {resume_checkpoint}")
    else:
        print("No checkpoint found — starting from scratch.")
else:
    print("No output dir found — starting from scratch.")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    dataloader_num_workers=4,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print(f"Starting training: {len(tokenized_dataset):,} samples, {EPOCHS} epochs, batch={BATCH_SIZE}")
trainer.train(resume_from_checkpoint=resume_checkpoint)

## 5. Save Adapter

In [ ]:
import os

adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

# Optional: push to HuggingFace Hub so you can load it in other notebooks
if HF_REPO_ID:
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"Pushed to HuggingFace Hub: {HF_REPO_ID}")

## 6. Quick Sanity Check — Does It Generate Code?
Before running full benchmarks, verify the fine-tuned model actually produces reasonable code.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Load fresh base + attach adapter
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
tuned_draft = PeftModel.from_pretrained(base, adapter_path)
tuned_draft.eval()

prompt = "def fibonacci(n):\n    "
inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = tuned_draft.generate(inputs, max_new_tokens=100, temperature=0.8, do_sample=True)

print("Fine-tuned draft output:")
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Next Step
Take the adapter path (`adapter_path`) and plug it into **Notebook 03** to run speculative decoding with the domain-tuned draft and compare acceptance rate vs the baseline.

In [ ]:
print(f"Adapter path for Notebook 03:\n  {os.path.abspath(adapter_path)}")